In [1]:
import asyncio
import textwrap
from typing import Any, Dict, Optional

def tool_function() -> str:
    return "test func called"

async def execute_dynamic_async(
    source_code: str, 
    global_vars: Optional[Dict[str, Any]] = None, 
    local_vars: Optional[Dict[str, Any]] = None
) -> Dict[str, Any]:
    if global_vars is None:
        global_vars = {}
    if local_vars is None:
        local_vars = {}

    wrapper_name = "_async_wrapper_func"
    wrapped_code = (
        f"async def {wrapper_name}():\n"
        f"{textwrap.indent(source_code, '    ')}\n"
        f"    return locals()"
    )

    exec_scope = global_vars.copy()
    exec_scope.update(local_vars)
    
    exec(wrapped_code, exec_scope)
    
    captured_locals = await exec_scope[wrapper_name]()
    
    local_vars.update(captured_locals)
    return local_vars

In [3]:
code = "result = tool_function(); print(result)"

globals_context = {
    "tool_function": tool_function,
    "print": print
}
locals_context = {}

await execute_dynamic_async(code, globals_context, locals_context)
print(locals_context)

test func called
{'result': 'test func called'}
